In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [3]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame
from sys import prefix
from listUtils import getFlatList
from musicdb import MusicDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM']
MasterParams()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb


# DB-Specific

In [4]:
from lib import deezer
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath("Deezer")
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Deezer API(Key=None)
Saving Perminant Deezer DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Deezer


# Master Files

In [5]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localRelatedData        = MusicDBData(path=permDir, fname="{0}SearchedForLocalRelatedData".format(db.lower()))
localRelated            = MusicDBData(path=permDir, fname="{0}SearchedForLocalRelated".format(db.lower()))
masterRelatedArtistData = mio.data.getRelatedArtistsData()
localArtistsInfoData    = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsInfoData".format(db.lower()))
localArtistsInfo        = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsInfo".format(db.lower()))
masterArtistsInfoData   = mio.data.getArtistsInfoData()
knownArtists            = mio.data.getSummaryNameData()
errors                  = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [6]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Related Artists:       {0}".format(len(localRelated.get())))
print("   Local Related Artists Data:  {0}".format(len(localRelatedData.get())))
print("   Master Related Artists Data: {0}".format(len(masterRelatedArtistData)))
print("   Local Artists Info:          {0}".format(len(localArtistsInfo.get())))
print("   Local Artists Info Data:     {0}".format(len(localArtistsInfoData.get())))
print("   Master Artists Info Data:    {0}".format(masterArtistsInfoData.shape[0]))
print("   Errors:                      {0}".format(len(errors.get())))
print("   Known Summary IDs:           {0}".format(len(knownArtists)))

Deezer Search Results
   Local Related Artists:       93208
   Local Related Artists Data:  0
   Master Related Artists Data: 62431
   Local Artists Info:          874919
   Local Artists Info Data:     0
   Master Artists Info Data:    874874
   Errors:                      1781
   Known Summary IDs:           1069560


# Download Artist Info Data

In [7]:
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData(debug=True)

Deezer API(Key=None)


## Find Artist IDs To Get

In [13]:
artistNames = mio.data.getSummaryNameData()
artistNames.name = "ArtistName"
basicData = DataFrame(artistNames).join(mio.data.getSummaryNumAlbumsData()).sort_values(by="NumAlbums", ascending=False)
localArtistsInfoDict = localArtistsInfo.get()
artistIDsToGet = basicData[~basicData.index.isin(localArtistsInfoDict.keys())]["ArtistName"]

print("{0} Search Results".format(db))
print("   Known Master Basic Data:   {0}".format(len(artistNames)))
print("   Known Artist Info Names:   {0}".format(len(localArtistsInfoDict)))
print("   Artist Names To Get:       {0}".format(len(artistIDsToGet)))

#   Artist Names To Get:       238653
#   Artist Names To Get:       204103

Deezer Search Results
   Known Master Basic Data:   1069560
   Known Artist Info Names:   893819
   Artist Names To Get:       185203


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "6:50am")
tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
searchedForLocalArtistsInfo     = localArtistsInfo.get()
searchedForLocalArtistsInfoData = localArtistsInfoData.get()
searchedForErrors               = errors.get()
for i,(artistID,artistName) in enumerate(artistIDsToGet.iteritems()):
    if searchedForLocalArtistsInfo.get(artistID) is not None:
        continue

    response = apiio.getArtistInfoData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        if response is None:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            apiio.sleep(3.5)
        else:
            searchedForLocalArtistsInfo[artistID] = "NoInfo"
            apiio.sleep(1.5)
        continue
    
    searchedForLocalArtistsInfo[artistID]     = artistName
    searchedForLocalArtistsInfoData[artistID] = response
    apiio.sleep(1.5)
    n += 1
        
    if n % 50 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), len(searchedForLocalArtistsInfoData), db))
        localArtistsInfo.save(data=searchedForLocalArtistsInfo)
        localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), db))
localArtistsInfo.save(data=searchedForLocalArtistsInfo)
localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

Process [Getting Deezer ArtistIDs] Start    ==> Time Is 2022-04-05 07:05:58
========================= termTime(day=today,time=7:00pm) =========================
   ====> Terminate Time Set To 2022-04-05 19:00:00 <====
   ====> Will Terminate Process 11 Hours and 54 Minutes From Now
Searching For Artist Info For Jin, Brain Crisis (1591407)                       	True
Searching For Artist Info For Rich Forbez (5398407)                             	True
Searching For Artist Info For C'funk (14218707)                                 	True
Searching For Artist Info For Nuff Sed, Joe Blast (1174607)                     	True
Searching For Artist Info For Hash24, Sopico & Lesram (13919807)                	True
Searching For Artist Info For Garth Jones (13379907)                            	True
Searching For Artist Info For Sara Ryan (10524907)                              	True
Searching For Artist Info For M.O.E Nino (12815607)                             	True
Searching For Artist Info For 

Searching For Artist Info For Group of Xhosa Men (6708707)                      	True
Searching For Artist Info For Enjoy Vs. DJ Schwede (1551707)                    	True
Searching For Artist Info For American Horror Story Collective (13226407)       	True
Searching For Artist Info For Jaden Chase (5231607)                             	True
Searching For Artist Info For Fatboy Slim & Koen Groeneveld (4959507)           	True
Searching For Artist Info For Luke Terry & Focus One (4420507)                  	True
Searching For Artist Info For Statik Majik (5527007)                            	True
Searching For Artist Info For G-20 (14125107)                                   	True
Searching For Artist Info For MR.T-Trax (6987507)                               	True
Searching For Artist Info For Bruno Costa (1413907)                             	True
Searching For Artist Info For Jônatas Ferreira (13665807)                      	True
Searching For Artist Info For Seb Tha Guru (14702607) 

Searching For Artist Info For The Skymasters and Maria Dieke-In (4532407)       	True
Searching For Artist Info For Substantial Error, Luke Truth (1623507)           	True
Searching For Artist Info For Jake Childs and Spettro (5860207)                 	True
Searching For Artist Info For Whispers of Andromeda (11562607)                  	True
Searching For Artist Info For Cheikha El Djenia Mascria (4620507)               	True
Searching For Artist Info For Clever & Durchtrieben (14217607)                  	True
Searching For Artist Info For Box Clever featuring Sarah Garvey (481107)        	True
Searching For Artist Info For Sean Cos Mason (14284707)                         	True
Searching For Artist Info For Tender Hooligans (403407)                         	True
Searching For Artist Info For Devito & Leon (14525107)                          	True
Searching For Artist Info For Eileen Khatchadourian (1418107)                   	True
Searching For Artist Info For Find the Beauty feat. Fl

Searching For Artist Info For K.O.F.A Lil' Louis Painting (11638307)            	True
Searching For Artist Info For Mic Geronimo Fearuring. Royal Flush (4107707)     	True
Searching For Artist Info For Morecambe And Wise (4256907)                      	True
Searching For Artist Info For Superlasciva (6708307)                            	True
Searching For Artist Info For K-yos (5706507)                                   	True
Searching For Artist Info For Smokealot (4381307)                               	True
Searching For Artist Info For Warren Zevon and Michael Wolff (6498307)          	True
Searching For Artist Info For Pusha Wood (13211207)                             	True
Searching For Artist Info For Máire Ní Chathasaigh, Chris Newman & Roy Whyke (1367707)	True
Searching For Artist Info For lovelesslust (1125207)                            	True
Searching For Artist Info For Wayne H. Freeman- Dan Gold fiddle (1065607)       	True
Searching For Artist Info For David Harness & 

Searching For Artist Info For Steve Harvey (149107)                             	True
Searching For Artist Info For CZM (6945807)                                     	True
Searching For Artist Info For Kurosuke (14106907)                               	True
Searching For Artist Info For Chucho Ferrer Y Su Organo (4851107)               	True
Searching For Artist Info For Suave Loc (1670007)                               	True
Searching For Artist Info For Darren Styles & Stonebank (13884207)              	True
Searching For Artist Info For Die Heimatsänger (176807)                        	True
Searching For Artist Info For Paolo Ciociola (290307)                           	True
Searching For Artist Info For Pancho El Junior & Fakuu Sanchez (11934807)       	True
Searching For Artist Info For Bobby One (5235007)                               	True
350/?      : Process [Getting Deezer ArtistIDs] Has Run For 12 Minutes and 59 Seconds.
Saving 894169 (New=350) Deezer Searched For Artist (I

...

Searching For Artist Info For Big Sleepy (6917511)                              	True
Searching For Artist Info For Jae Starz (2042011)                               	True
Searching For Artist Info For Loisan, Affkt (4093711)                           	True
Searching For Artist Info For Decca Symphony Orchestra, Eva Jessy Choir, Alexander Smallens (4960911)	True
Searching For Artist Info For Ben Betts (14145311)                              	True
Searching For Artist Info For Jimmy Bracken (2736111)                           	True
Searching For Artist Info For Szu Kuo-Lan & Lu Yuan-Che & Tsai Fu-Kuei (13492011)	True
Searching For Artist Info For Los Cojones Del Toro (10607811)                   	True
Searching For Artist Info For Korgbrain (1738011)                               	True
Searching For Artist Info For Tiny Das Neves (13189611)                         	True
11800/?    : Process [Getting Deezer ArtistIDs] Has Run For 7 Hours and 16 Minutes.
Saving 905619 (New=11800) Deezer S

Searching For Artist Info For Pang Lozano (13703211)                            	True
Searching For Artist Info For Kaslu LK (13205111)                               	True
Searching For Artist Info For Ramadan Krasniqi Dani (11618411)                  	True
Searching For Artist Info For LVSS!! (13725611)                                 	True
Searching For Artist Info For Jah Ghatti (3585411)                              	True
Searching For Artist Info For King DJ (3783911)                                 	True
Searching For Artist Info For Karolina Eiriksdottir (3019711)                   	True
Searching For Artist Info For IYB Lach (14009411)                               	True
Searching For Artist Info For Łukasz Lach (5281811)                             	True
Searching For Artist Info For LiL TExAS x REWROTE (4721211)                     	True
Searching For Artist Info For Arthur Salles (10660611)                          	True
Searching For Artist Info For DJ Nylezzz (1888111)    

Searching For Artist Info For Namie Amuro (2966411)                             	True
Searching For Artist Info For Ese Deleon (12739811)                             	True
Searching For Artist Info For Karl Roy (5617911)                                	True
Searching For Artist Info For Eva Plankova, Maria Hanakovicova & Gary Cox (3655911)	True
Searching For Artist Info For 3yce (15365511)                                   	True
Searching For Artist Info For Diesel FX (12624411)                              	True
Searching For Artist Info For Kid Vicious & Nick K. (2595311)                   	True
Searching For Artist Info For Rhadow, Mirco Violi, Matt Brown (2270111)         	True
Searching For Artist Info For Musicanova (203911)                               	True
Searching For Artist Info For Mad Rush (5210011)                                	True
Searching For Artist Info For Wooks (48947611)                                  	True
Searching For Artist Info For The Shark (985411)   

Searching For Artist Info For Jason Country (142689412)                         	True
Searching For Artist Info For Xavier Batllés, Salvador Font, Carles Benavent, Lluís Cabanach, Joan Albert Amargós & Jaume Cortadellas (7863812)	True
Searching For Artist Info For PM Capo (85568912)                                	True
Searching For Artist Info For Ray Holmes (57986912)                             	True
Searching For Artist Info For Smokin' Iris (54546712)                           	True
Searching For Artist Info For Ray-Jay Band (52320312)                           	True
12050/?    : Process [Getting Deezer ArtistIDs] Has Run For 7 Hours and 25 Minutes.
Saving 905869 (New=12050) Deezer Searched For Artist (Info) IDs
Searching For Artist Info For Jay Dub (317612)                                  	True
Searching For Artist Info For Human Calling (152757812)                         	True
Searching For Artist Info For Edu y Jampy (117348912)                           	True
Searching Fo

Searching For Artist Info For DJ Prince Eskhosini (60809312)                    	True
Searching For Artist Info For Peedi Crakk (992612)                              	True
Searching For Artist Info For Bree2ez (57156512)                                	True
Searching For Artist Info For Rich Sole (151285212)                             	True
Searching For Artist Info For Matan Caspi, D-Formation (130455212)              	True
Searching For Artist Info For D-Formation & Sean & Dee (148615912)              	True
Searching For Artist Info For Groove Riders (175112)                            	True
Searching For Artist Info For Riddim Doctors (88305812)                         	True
Searching For Artist Info For K. Brit (127143212)                               	True
Searching For Artist Info For Reshboy (11034812)                                	True
Searching For Artist Info For Matt Caseli (150512)                              	True
Searching For Artist Info For Chris Eckman and The Las

## Save Results

In [10]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(laid):
    df = None
    if isinstance(laid,dict) and len(laid) > 0:
        df = DataFrame(laid.values())
    return df
        
def getResponseDataFrame(laid):
    df = getArtistNamesDataFrame(laid)
    if not isinstance(df,DataFrame):
        return None
    df.index = df['id']
    df.drop(['id'], axis=1, inplace=True)
    return df

In [11]:
from pandas import concat
laid = localArtistsInfoData.get()
df  = getResponseDataFrame(laid)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getArtistsInfoData()    
    prevNewArtists = len(searchDF[~searchDF.index.isin(knownArtists.index)])
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['fans'] = searchDF['fans'].fillna(0).astype(int)
    searchDF = searchDF.sort_values(by='fans', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(len(searchDF[searchDF.index.isin(knownArtists.index)])))
    print("  ==> {0} New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])))
    print("  ==> {0} Delta New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])-prevNewArtists))
    print("Saving Data")
    mio.data.saveArtistsInfoData(data=searchDF)
else:
    print("No new artists")

Found 18900 New Artists
Found 874874 Previous Artists
Found 893774 Total Artists
Found 893773 Unique Artists
Found 893773 Unique + Sorted Artists
  ==> 345640 Old Artists
  ==> 548133 New Artists
  ==> 18899 Delta New Artists
Saving Data


In [12]:
localArtistsInfoData.save(data={})

# Download Related Artist Search Data

In [ ]:
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData(debug=True)

## Find Related Artists

In [ ]:
useBasicData       = False
useSelfRelatedData = True
useMasterIDs       = False

if useBasicData is True:
    knownRelatedArtists = localRelated.get()
    basicData = DataFrame(mio.data.getSummaryNameData()).join(mio.data.getSummaryNumAlbumsData()).sort_values(by="NumAlbums", ascending=False)
    basicData.columns = ["ArtistName", "NumAlbums"]
    artistRelatedToGet = basicData["ArtistName"][~basicData["ArtistName"].index.isin(knownRelatedArtists.keys())]

    print("{0} Search Results".format(db))
    print("   Known Master Basic Data:     {0}".format(basicData.shape[0]))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))
elif useSelfRelatedData is True:
    artistRelatedToGet  = {}
    knownRelatedArtists = localRelated.get()
    masterRelatedArtistData = mio.data.getRelatedArtistsData()
    for artistID,artistIDData in masterRelatedArtistData.iteritems():
        artistRelatedToGet.update({str(k): v for k,v in artistIDData.items() if knownRelatedArtists.get(str(k)) is None})
    artistRelatedToGet = Series(artistRelatedToGet)

    print("{0} Search Results".format(db))
    print("   Known Master Related Data:   {0}".format(len(masterRelatedArtistData)))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))
elif useMasterIDs is True:
    meu = MusicDBIO()
    mmeDF = meu.getData()
    deezerMatchedArtistsData = mmeDF[mmeDF["Deezer"].notna()][["ArtistName", "Deezer"]]
    deezerMatchedArtists = deezerMatchedArtistsData["ArtistName"].copy(deep=True)
    deezerMatchedArtists.index = deezerMatchedArtistsData["Deezer"]
    artistRelatedToGet = Series({artistID: artistName for artistID,artistName in deezerMatchedArtists.iteritems() if knownRelatedArtists.get(artistID) is None})
    print("{0} Search Results".format(db))
    print("   Known Master Related Data:   {0}".format(len(deezerMatchedArtists)))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "7:36pm")
tt = TermTime("today", "10:15pm")
maxN = 50000000

n  = 0
searchedForLocalRelated         = localRelated.get()
searchedForLocalRelatedData     = localRelatedData.get()
searchedForLocalArtistsInfo     = localArtistsInfo.get()
searchedForLocalArtistsInfoData = localArtistsInfoData.get()
searchedForErrors               = errors.get()
for i,(artistID,artistName) in enumerate(artistRelatedToGet.iteritems()):
    if searchedForLocalRelated.get(artistID) is not None:
        continue

    response = apiio.getArtistRelatedData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        if response is None:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            searchedForLocalRelated[artistID] = artistName
            apiio.sleep(3.5)
        else:
            searchedForLocalRelated[artistID] = artistName
            apiio.sleep(1.5)
        continue
    
    searchedForLocalRelated[artistID]     = artistName
    searchedForLocalRelatedData[artistID] = {str(rec['id']): rec['name'] for rec in response}
    for record in response:
        recID = str(record['id'])
        if searchedForLocalArtistsInfo.get(recID) is None:
            searchedForLocalArtistsInfo[recID]     = artistName
            searchedForLocalArtistsInfoData[recID] = record
    apiio.sleep(1.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Related) IDs".format(len(searchedForLocalRelated), db))
        localRelated.save(data=searchedForLocalRelated)
        localRelatedData.save(data=searchedForLocalRelatedData)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), len(searchedForLocalArtistsInfoData), db))
        localArtistsInfo.save(data=searchedForLocalArtistsInfo)
        localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break
    
ts.stop()
print("Saving {0} {1} Searched For Artist (Related) IDs".format(len(searchedForLocalRelated), db))
localRelated.save(data=searchedForLocalRelated)
localRelatedData.save(data=searchedForLocalRelatedData)
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), db))
localArtistsInfo.save(data=searchedForLocalArtistsInfo)
localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(laid):
    df = None
    if isinstance(laid,dict) and len(laid) > 0:
        df = DataFrame(laid.values())
    return df
        
def getResponseDataFrame(laid):
    df = getArtistNamesDataFrame(laid)
    if not isinstance(df,DataFrame):
        return None
    df.index = df['id']
    df.drop(['id'], axis=1, inplace=True)
    return df

In [ ]:
laid = localArtistsInfoData.get()
df  = getResponseDataFrame(laid)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getArtistsInfoData()    
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['fans'] = searchDF['fans'].astype(int)
    searchDF = searchDF.sort_values(by='fans', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveArtistsInfoData(data=searchDF)
else:
    print("No new artists")

In [ ]:
df = localRelatedData.get()
if isinstance(df,dict):
    print("Found {0} New Artists".format(len(df)))
    searchDF = mio.data.getRelatedArtistsData()
    if isinstance(searchDF,Series):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = searchDF.append(Series(df))
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveRelatedArtistsData(data=searchDF)

In [ ]:
localRelatedData.save(data={})
localArtistsInfoData.save(data={})

# Combined Artist Info Data

In [ ]:
artistRelatedData = mio.data.getRelatedArtistsData()
artistRelatedData.name = "RelatedArtists"
artistRelatedData = DataFrame(artistRelatedData)
artistInfoData    = mio.data.getArtistsInfoData()

In [ ]:
artistInfoData

In [ ]:
from pandas import merge
artistInfoData.join(artistRelatedData)

In [ ]:
knownArtists